# Week 5 Inventory Analytics

This notebook answers one direct question: is stock at the right level, too much, or too little? We review turnover, stock health, aging, ABC classification, and the amount of stock value exposed to risk.

Important rules used:
- Use `current_stock`, `reorder_level`, `safety_stock`, and `max_stock` from `inventory_master` to define stock health.
- Use the product-branch average stock formula for turnover: units sold divided by average stock.
- Do not use `running_balance` for aging or balance checks. This week uses movement dates and quantities instead.
- The 9x issue is investigated using the actual revenue definition in the dataset.

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

root = Path.cwd()
while not (root / 'data').is_dir() and root != root.parent:
    root = root.parent
if not (root / 'data').is_dir():
    raise FileNotFoundError('Could not locate the project data directory from the notebook working directory.')
sales = pd.read_csv(root / 'data' / 'processed' / 'sales_table.csv', parse_dates=['order_date', 'delivery_date', 'invoice_date', 'due_date', 'last_payment_date'])
inventory = pd.read_csv(root / 'data' / 'cleaned' / 'inventory_master.csv')
products = pd.read_csv(root / 'data' / 'cleaned' / 'products.csv', parse_dates=['last_purchase_date'])
stock = pd.read_csv(root / 'data' / 'processed' / 'purchases_fact_table.csv', parse_dates=['movement_date'])

print('Sales rows:', len(sales))
print('Inventory rows:', len(inventory))
print('Stock movements:', len(stock))
print('Products:', len(products))

In [ ]:
sales_by_pb = sales.groupby(['product_id', 'branch_id'], as_index=False)['quantity'].sum().rename(columns={'quantity': 'units_sold'})
inventory_ext = inventory.merge(products[['product_id', 'product_name', 'category', 'unit_cost']], on='product_id', how='left')
turnover = sales_by_pb.merge(
    inventory_ext[['product_id', 'branch_id', 'opening_stock', 'current_stock', 'reorder_level', 'safety_stock', 'max_stock', 'unit_cost']],
    on=['product_id', 'branch_id'],
    how='left'
)
turnover['average_stock'] = (turnover['opening_stock'] + turnover['current_stock']) / 2
turnover['turnover_ratio'] = turnover['units_sold'] / turnover['average_stock'].replace(0, pd.NA)
display(turnover.sort_values('turnover_ratio', ascending=False).head(10)[['product_id', 'branch_id', 'units_sold', 'current_stock', 'average_stock', 'turnover_ratio']])


### What this means

Turnover tells us how often stock is being sold versus how much is sitting in the warehouse. A product with a high turnover ratio is moving quickly and is likely healthy. A lower ratio means inventory is staying on the shelf for longer than expected.

In [ ]:
def classify_health(row):
    if row['current_stock'] > row['max_stock']:
        return 'Overstocked'
    if row['current_stock'] < row['safety_stock']:
        return 'Critical'
    if row['current_stock'] < row['reorder_level']:
        return 'Low'
    return 'Healthy'

health = turnover.copy()
health['stock_health'] = health.apply(classify_health, axis=1)
print(health['stock_health'].value_counts().to_string())
display(health[health['stock_health'] != 'Healthy'][['product_id', 'branch_id', 'current_stock', 'reorder_level', 'safety_stock', 'max_stock', 'stock_health']].head(15))


### What this means

This health check tells us whether each product-branch pair is above target, below the reorder point, or in danger of a stockout. It is a simple rules-based way to flag where operational attention is needed.

In [ ]:
last_movement = stock.groupby(['product_id', 'branch_id'], as_index=False)['movement_date'].max().rename(columns={'movement_date': 'last_movement_date'})
movement_total = stock.groupby(['product_id', 'branch_id'], as_index=False)['quantity'].sum().rename(columns={'quantity': 'total_quantity_moved'})
aging = inventory_ext[['product_id', 'branch_id', 'current_stock', 'unit_cost', 'product_name', 'category']].merge(last_movement, on=['product_id', 'branch_id'], how='left').merge(movement_total, on=['product_id', 'branch_id'], how='left')
aging['days_since_last_movement'] = (stock['movement_date'].max() - aging['last_movement_date']).dt.days
aging['stock_value'] = aging['current_stock'] * aging['unit_cost']
display(aging.sort_values('days_since_last_movement', ascending=False).head(10)[['product_name', 'category', 'branch_id', 'current_stock', 'days_since_last_movement', 'stock_value']])


### What this means

Aging is the signal for "how long has this stock been sitting around?" We are using the last movement date rather than `running_balance` because the balance values do not reconcile reliably. That is a more honest way to assess age in this dataset.

In [ ]:
        product_revenue = sales.groupby('product_id', as_index=False)['line_grand_total'].sum().rename(columns={'line_grand_total': 'revenue'})
        abc_df = product_revenue.merge(products[['product_id', 'product_name', 'category']], on='product_id', how='left')
        abc_df = abc_df.sort_values('revenue', ascending=False).reset_index(drop=True)
        abc_df['cumulative_revenue'] = abc_df['revenue'].cumsum()
        abc_df['cumulative_pct'] = abc_df['cumulative_revenue'] / abc_df['revenue'].sum()
        abc_df['abc_class'] = 'C'
        abc_df.loc[abc_df['cumulative_pct'] <= 0.80, 'abc_class'] = 'A'
        abc_df.loc[(abc_df['cumulative_pct'] > 0.80) & (abc_df['cumulative_pct'] <= 0.95), 'abc_class'] = 'B'
        display(abc_df.head(10)[['product_name', 'category', 'revenue', 'cumulative_pct', 'abc_class']])
        print('ABC counts:
', abc_df['abc_class'].value_counts().to_string())


### What this means

The ABC view is a concentration check. In most businesses, a small number of products generate most of the revenue, and the same pattern shows up here. It helps focus attention where the money is coming from.

In [ ]:
risk = inventory_ext[(inventory_ext['current_stock'] > inventory_ext['max_stock']) | (inventory_ext['current_stock'] < inventory_ext['safety_stock'])].copy()
risk['risk_value'] = risk['current_stock'] * risk['unit_cost']
risk['risk_type'] = risk.apply(lambda row: 'Overstocked' if row['current_stock'] > row['max_stock'] else 'Understocked', axis=1)
risk_table = risk[['product_id', 'product_name', 'category', 'branch_id', 'current_stock', 'safety_stock', 'max_stock', 'risk_type', 'risk_value']].sort_values('risk_value', ascending=False)
display(risk_table.head(15))
print('Total stock value in risk items:', risk_table['risk_value'].sum())


### The 9x stock value signal

We tested the stock-value-to-revenue question using the actual data. If we compare current stock value to product-line revenue, the ratio is about 8.7x. If we compare to order-level revenue, the ratio is about 1.0x. The difference matters because the project contains both line-level sales values and order-level totals. This means the stock size is not automatically "wrong" in every metric; it is high relative to line revenue, but much closer to order revenue when the broader sales total is used.

In [ ]:
stock_value_total = inventory_ext['current_stock'].mul(inventory_ext['unit_cost']).sum()
line_revenue_total = sales['line_grand_total'].sum()
order_revenue_total = sales['grand_total'].sum()
print('Current stock value:', stock_value_total)
print('Line revenue total:', line_revenue_total)
print('Order revenue total:', order_revenue_total)
print('Ratio stock value to line revenue:', stock_value_total / line_revenue_total)
print('Ratio stock value to order revenue:', stock_value_total / order_revenue_total)


### Final takeaways

This Week 5 view shows that the warehouse is carrying a large stock base, and the risk is concentrated in a small number of product-branch situations. The healthy path is to maintain the products that sell quickly and to watch the items where stock sits above the maximum or below the safe floor. The key takeaway is that the stock issue is real, but the exact strength of the overstock signal depends on which revenue measure is being used.